# Scheduler Version Benchmark Workbench

This notebook builds every registered scheduler version, runs each one against the same
policy-independent scenario suite, and compares legality and performance. It is the
experiment dashboard for the optimization sequence developed in the companion
`edge_cloud_scheduling_lab.ipynb` notebook. The registry also includes rejected v84–v88 research
revisions so their regressions remain visible beside the promoted policy.

## Goal

Use one repeatable loop for every scheduler iteration:

1. freeze a named source version;
2. compile all selected versions with the same flags;
3. run the same selected scenarios through the dynamic local judge;
4. reject illegal policies before interpreting performance;
5. compare scenario-level score, throughput, TDR, TPOT, and elapsed time; and
6. save an optional durable experiment record.

**Decision rule:** score is the primary local outcome, but a policy should not be accepted
until its scenario-level tradeoffs are understood. An unweighted suite mean is a convenient
summary, not a substitute for the official hidden tests.

## Setup

In [1]:
from __future__ import annotations

import hashlib
import html
import json
import math
import os
import re
import shlex
import subprocess
from pathlib import Path
from typing import Any, Iterable

from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "main.cpp").is_file() and (candidate / "tools/local_judge.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find the scheduler repository")


REPO_ROOT = find_repo_root()
REGISTRY_PATH = REPO_ROOT / "scheduler_versions/registry.json"
SCENARIO_DIR = REPO_ROOT / "scenarios"
REFERENCE_RESULTS_PATH = REPO_ROOT / "benchmarks/baseline-v0.json"
RUN_DIR = REPO_ROOT / "build/benchmark-runs"
BIN_DIR = REPO_ROOT / "build/scheduler-versions"

RUN_DIR.mkdir(parents=True, exist_ok=True)
BIN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Registry:   {REGISTRY_PATH.relative_to(REPO_ROOT)}")

Repository: /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling
Registry:   scheduler_versions/registry.json


In [2]:
def display_table(
    rows: Iterable[dict[str, Any]],
    columns: list[tuple[str, str]] | None = None,
    cell_style: Any | None = None,
) -> None:
    bounded_rows = list(rows)
    if not bounded_rows:
        display(Markdown("_No rows._"))
        return
    if columns is None:
        columns = [(key, key) for key in bounded_rows[0]]
    header = "".join(f"<th>{html.escape(label)}</th>" for _, label in columns)
    body = []
    for row in bounded_rows:
        cells = []
        for key, _ in columns:
            style = cell_style(row, key) if cell_style else ""
            cells.append(
                f'<td style="{html.escape(style)}">{html.escape(str(row.get(key, "")))}</td>'
            )
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(
        HTML(
            "<table><thead><tr>"
            + header
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table>"
        )
    )


def run_command(command: list[str], timeout_seconds: float = 180.0) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
    )
    if completed.returncode != 0:
        detail = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(command)}\n{detail}")
    return completed

## 1. Select versions and scenarios

`all` is the normal comparison. Replace it with a list of names for a faster focused run.
The benchmark is deterministic, so repeated runs are a reproducibility check rather than a
statistical sample.

In [3]:
VERSIONS_TO_RUN: str | list[str] = "all"
SCENARIOS_TO_RUN: str | list[str] = "all"
REFERENCE_VERSION = "v0-baseline"
FOCUS_SCENARIO = "batch_friendly_burst"

# Optional durable save. The normal outputs under build/ are disposable and ignored by Git.
SAVE_DURABLE_RUN = False
DURABLE_RUN_LABEL = "replace-with-experiment-name"

# Display-only thresholds. They flag tradeoffs but do not fail the notebook.
SCORE_REGRESSION_ALERT = -1.0
THROUGHPUT_REGRESSION_ALERT_PERCENT = -5.0
LATENCY_REGRESSION_ALERT_PERCENT = 10.0

In [4]:
registry = json.loads(REGISTRY_PATH.read_text())
assert registry["schema_version"] == 1
registered_versions = registry["versions"]

version_names = [version["name"] for version in registered_versions]
assert len(version_names) == len(set(version_names)), "Scheduler names must be unique"
assert REFERENCE_VERSION in version_names

if VERSIONS_TO_RUN == "all":
    selected_versions = registered_versions
else:
    requested_versions = set(VERSIONS_TO_RUN)
    unknown_versions = requested_versions.difference(version_names)
    assert not unknown_versions, f"Unknown versions: {sorted(unknown_versions)}"
    selected_versions = [
        version for version in registered_versions if version["name"] in requested_versions
    ]
    if REFERENCE_VERSION not in requested_versions:
        selected_versions.insert(
            0, next(version for version in registered_versions if version["name"] == REFERENCE_VERSION)
        )

version_rows = []
for version in selected_versions:
    source_path = REPO_ROOT / version["source"]
    assert source_path.is_file(), f"Missing source: {source_path}"
    compile_defines = version.get("compile_defines", [])
    hash_input = source_path.read_bytes() + json.dumps(compile_defines, sort_keys=True).encode()
    source_hash = hashlib.sha256(hash_input).hexdigest()
    version_rows.append(
        {
            "version": version["name"],
            "layer": version.get("layer", 0),
            "source": version["source"],
            "defines": ", ".join(compile_defines) or "none",
            "frozen": version["frozen"],
            "sha256": source_hash[:12],
            "description": version["description"],
        }
    )

display_table(version_rows)

version,layer,source,defines,frozen,sha256,description
v0-baseline,0,scheduler_versions/v0_baseline.cpp,none,True,881e36158115,Frozen FIFO singleton baseline before scoring optimizations.
v1-multi-active,1,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=1,True,6f48f3a8620e,Multiple unfinished singleton requests per cloud with round-robin assignment.
v2-load-aware,2,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=2,True,05cd4859b0e4,"Adds cloud assignment using busy time, known prefill work, ready decode work, and an active-request proxy."
v3-immediate-groups,3,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=3,True,8a9b9d26283e,"Adds immediate grouping for ready D PRE, same-cloud D PROC, and D POST work."
v4-table-groups,4,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=4,True,60cb4d3fb408,Selects decode group size by interpolated task-table service rate instead of always taking every ready member.
v5-slo-aware,5,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=5,True,51d6b700542d,Adds conservative SLO urgency and tightly gated controlled waiting when future events are known.
v6-prefill-chunks,6,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=6,True,b108679d9b56,Adds adaptive gap-free P PROC layer chunks when longer prefills compete with other cloud work.
v7-link-aware,7,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=7,True,d0c67ee81a2c,Adds transfer-aware group cost and latency-weighted shortest-prefill ordering while preserving FIFO on throughput-weighted links.
v8-exact-timelines,8,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=8,True,7129af91919d,Tracks exact virtual resource and FIFO-link finish times and bounds batching waits by the next known event.
v9-fanout-cohorts,9,scheduler_versions/layered_scheduler.cpp,OPT_LEVEL=9,True,c89949fc4a4b,Adds cloud-fanout-aware D PRE group formation and waits only for compatible decode cohort events.


In [5]:
all_scenario_paths = sorted(SCENARIO_DIR.glob("*.json"))
scenario_data = {json.loads(path.read_text())["name"]: json.loads(path.read_text()) for path in all_scenario_paths}
scenario_path_by_name = {
    json.loads(path.read_text())["name"]: path for path in all_scenario_paths
}

if SCENARIOS_TO_RUN == "all":
    selected_scenario_names = list(scenario_path_by_name)
else:
    selected_scenario_names = list(SCENARIOS_TO_RUN)
    unknown_scenarios = set(selected_scenario_names).difference(scenario_path_by_name)
    assert not unknown_scenarios, f"Unknown scenarios: {sorted(unknown_scenarios)}"

display_table(
    [
        {
            "scenario": name,
            "requests": len(scenario_data[name]["requests"]),
            "tokens": sum(request["output_length"] for request in scenario_data[name]["requests"]),
            "throughput weight": scenario_data[name]["scoring"]["w_tp"],
            "latency weight": scenario_data[name]["scoring"]["w_c"],
        }
        for name in selected_scenario_names
    ]
)

scenario,requests,tokens,throughput weight,latency weight
official_worked_example,1,1,0.5,0.5
single_sanity,1,3,0.5,0.5
two_cloud_parallel,6,29,0.5,0.5
output_length_skew,6,56,0.6,0.4
batch_friendly_burst,16,128,0.9,0.1
latency_sensitive_stream,12,60,0.2,0.8
link_bottleneck,8,32,0.6,0.4
prefill_preemption,5,64,0.5,0.5
degenerate_one_layer,3,6,0.5,0.5
interpolation_missing_values,4,10,0.5,0.5


## 2. Build every selected version

Every source is compiled independently with identical base flags. The displayed fingerprint
hashes both source bytes and compile definitions, so two feature levels remain distinguishable
even though they use the same C++ file.

In [6]:
CXX = os.environ.get("CXX", "g++")
CXXFLAGS = shlex.split(
    os.environ.get("CXXFLAGS", "-std=c++17 -O2 -pipe -Wall -Wextra -Wpedantic")
)

executables: dict[str, Path] = {}
build_rows = []
for version in selected_versions:
    name = version["name"]
    source_path = REPO_ROOT / version["source"]
    executable = BIN_DIR / name
    define_flags = [f"-D{define}" for define in version.get("compile_defines", [])]
    completed = run_command(
        [CXX, *CXXFLAGS, *define_flags, str(source_path), "-o", str(executable)]
    )
    executables[name] = executable
    build_rows.append(
        {
            "version": name,
            "status": "PASS",
            "compiler": CXX,
            "warnings": len([line for line in completed.stderr.splitlines() if "warning:" in line]),
            "executable": executable.relative_to(REPO_ROOT),
        }
    )

display_table(build_rows)

version,status,compiler,warnings,executable
v0-baseline,PASS,g++,0,build/scheduler-versions/v0-baseline
v1-multi-active,PASS,g++,0,build/scheduler-versions/v1-multi-active
v2-load-aware,PASS,g++,0,build/scheduler-versions/v2-load-aware
v3-immediate-groups,PASS,g++,0,build/scheduler-versions/v3-immediate-groups
v4-table-groups,PASS,g++,0,build/scheduler-versions/v4-table-groups
v5-slo-aware,PASS,g++,0,build/scheduler-versions/v5-slo-aware
v6-prefill-chunks,PASS,g++,0,build/scheduler-versions/v6-prefill-chunks
v7-link-aware,PASS,g++,0,build/scheduler-versions/v7-link-aware
v8-exact-timelines,PASS,g++,0,build/scheduler-versions/v8-exact-timelines
v9-fanout-cohorts,PASS,g++,0,build/scheduler-versions/v9-fanout-cohorts


The fixed transcript tests assert the baseline's exact output order. A future optimization
may produce a different but legal transcript, so those tests are **not** used as a universal
policy gate. The dynamic local judge below validates legal decisions regardless of policy.

## 3. Benchmark every version

In [7]:
scenario_arguments = [str(scenario_path_by_name[name]) for name in selected_scenario_names]
results_by_version: dict[str, list[dict[str, Any]]] = {}
run_rows = []

for version in selected_versions:
    name = version["name"]
    output_path = RUN_DIR / f"{name}.json"
    command = [
        "python3",
        "tools/local_judge.py",
        "--solver",
        str(executables[name]),
        "--scenarios",
        *scenario_arguments,
        "--json-out",
        str(output_path),
    ]
    completed = run_command(command)
    results = json.loads(output_path.read_text())
    results_by_version[name] = results
    run_rows.append(
        {
            "version": name,
            "legal scenarios": sum(bool(row.get("legal")) for row in results),
            "scenario count": len(results),
            "status": "PASS" if all(row.get("legal") for row in results) else "ILLEGAL",
        }
    )

display_table(run_rows)

version,legal scenarios,scenario count,status
v0-baseline,29,29,PASS
v1-multi-active,29,29,PASS
v2-load-aware,29,29,PASS
v3-immediate-groups,29,29,PASS
v4-table-groups,29,29,PASS
v5-slo-aware,29,29,PASS
v6-prefill-chunks,29,29,PASS
v7-link-aware,29,29,PASS
v8-exact-timelines,29,29,PASS
v9-fanout-cohorts,29,29,PASS


## 4. Validate the benchmark inputs and calculations

Validation is performed before rankings are displayed:

- each version must return one result for every selected scenario;
- local-judge legality is mandatory;
- token counts must match scenario truth;
- score components are recomputed independently; and
- the frozen `v0-baseline` source must reproduce `baseline-v0.json` on the full suite.

In [8]:
def clamp01(value: float) -> float:
    return max(0.0, min(1.0, value))


def recompute_score(result: dict[str, Any], scoring: dict[str, Any]) -> float:
    excess_tdr = max(0.0, (result["tdr"] - scoring["SLO1"]) / scoring["SLO1"])
    excess_tpot = max(0.0, (result["tpot"] - scoring["SLO2"]) / scoring["SLO2"])
    distance = math.hypot(excess_tdr, excess_tpot)
    throughput_component = clamp01(
        (result["throughput"] - scoring["tp_base"])
        / (scoring["tp_UB"] - scoring["tp_base"])
    )
    distance_base = scoring["dist_base"]
    latency_component = (
        max(0.0, 1.0 - distance / distance_base)
        if distance_base > 0
        else (1.0 if distance == 0 else 0.0)
    )
    return 1000.0 * (
        scoring["w_tp"] * throughput_component + scoring["w_c"] * latency_component
    )


validation_rows = []
for version_name, results in results_by_version.items():
    result_names = [result["scenario"] for result in results]
    assert len(result_names) == len(set(result_names))
    assert set(result_names) == set(selected_scenario_names)
    assert all(result.get("legal") for result in results), f"{version_name} produced an illegal schedule"

    maximum_score_error = 0.0
    for result in results:
        scenario = scenario_data[result["scenario"]]
        expected_tokens = sum(request["output_length"] for request in scenario["requests"])
        assert result["tokens"] == expected_tokens
        recalculated = recompute_score(result, scenario["scoring"])
        maximum_score_error = max(maximum_score_error, abs(recalculated - result["score"]))
        assert 0.0 <= result["score"] <= 1000.0
        assert result["throughput"] > 0.0 and result["elapsed"] > 0.0
        assert result["tdr"] >= 0.0 and result["tpot"] >= 0.0
    assert maximum_score_error < 1e-7
    validation_rows.append(
        {
            "version": version_name,
            "coverage": f"{len(results)}/{len(selected_scenario_names)}",
            "legal": "yes",
            "token counts": "verified",
            "max score error": f"{maximum_score_error:.2e}",
        }
    )

display_table(validation_rows)

version,coverage,legal,token counts,max score error
v0-baseline,29/29,yes,verified,0.00e+00
v1-multi-active,29/29,yes,verified,0.00e+00
v2-load-aware,29/29,yes,verified,0.00e+00
v3-immediate-groups,29/29,yes,verified,0.00e+00
v4-table-groups,29/29,yes,verified,0.00e+00
v5-slo-aware,29/29,yes,verified,0.00e+00
v6-prefill-chunks,29/29,yes,verified,0.00e+00
v7-link-aware,29/29,yes,verified,0.00e+00
v8-exact-timelines,29/29,yes,verified,0.00e+00
v9-fanout-cohorts,29/29,yes,verified,0.00e+00


In [9]:
if set(selected_scenario_names) == set(scenario_path_by_name):
    expected_baseline = {
        row["scenario"]: row for row in json.loads(REFERENCE_RESULTS_PATH.read_text())
    }
    observed_baseline = {
        row["scenario"]: row for row in results_by_version[REFERENCE_VERSION]
    }
    maximum_snapshot_delta = max(
        abs(observed_baseline[name][metric] - expected[metric])
        for name, expected in expected_baseline.items()
        for metric in ("score", "throughput", "tdr", "tpot", "elapsed")
    )
    assert maximum_snapshot_delta < 1e-6
    print(f"Frozen baseline reproduced baseline-v0.json; max metric delta={maximum_snapshot_delta:.2e}")
else:
    maximum_snapshot_delta = None
    print("Baseline snapshot check skipped because this is a filtered scenario run.")

Frozen baseline reproduced baseline-v0.json; max metric delta=0.00e+00


## 5. Suite results

The summary treats every selected scenario equally. Relative throughput, latency, and elapsed
columns are geometric means of per-scenario ratios to the reference. This avoids pretending
that raw throughput values from different workloads share one denominator.
`scheduler CPU ms` is the child-process CPU consumed across the selected cases. It is a coarse
complexity guard; these short interactions are not a precision microbenchmark.

In [10]:
def result_map(version_name: str) -> dict[str, dict[str, Any]]:
    return {row["scenario"]: row for row in results_by_version[version_name]}


reference_map = result_map(REFERENCE_VERSION)


def geometric_mean_ratio(ratios: list[float]) -> float:
    assert ratios and all(ratio > 0 for ratio in ratios)
    return math.exp(sum(math.log(ratio) for ratio in ratios) / len(ratios))


def format_geometric_change(ratios: list[float]) -> str:
    if not ratios:
        return "n/a"
    return f"{100 * (geometric_mean_ratio(ratios) - 1):+.1f}%"


summary_rows = []
for version in selected_versions:
    name = version["name"]
    current = result_map(name)
    scores = [current[scenario]["score"] for scenario in selected_scenario_names]
    score_deltas = [
        current[scenario]["score"] - reference_map[scenario]["score"]
        for scenario in selected_scenario_names
    ]
    tp_ratios = [
        current[scenario]["throughput"] / reference_map[scenario]["throughput"]
        for scenario in selected_scenario_names
    ]
    tdr_ratios = [
        current[scenario]["tdr"] / reference_map[scenario]["tdr"]
        for scenario in selected_scenario_names
        if reference_map[scenario]["tdr"] > 0 and current[scenario]["tdr"] > 0
    ]
    tpot_ratios = [
        current[scenario]["tpot"] / reference_map[scenario]["tpot"]
        for scenario in selected_scenario_names
        if reference_map[scenario]["tpot"] > 0 and current[scenario]["tpot"] > 0
    ]
    elapsed_ratios = [
        current[scenario]["elapsed"] / reference_map[scenario]["elapsed"]
        for scenario in selected_scenario_names
    ]
    scheduler_cpu_ms = 1000.0 * sum(
        current[scenario].get("scheduler_cpu_seconds", 0.0)
        for scenario in selected_scenario_names
    )
    summary_rows.append(
        {
            "version": name,
            "mean score": f"{sum(scores) / len(scores):.3f}",
            "mean score delta": f"{sum(score_deltas) / len(score_deltas):+.3f}",
            "wins / ties / losses": (
                f"{sum(delta > 1e-9 for delta in score_deltas)} / "
                f"{sum(abs(delta) <= 1e-9 for delta in score_deltas)} / "
                f"{sum(delta < -1e-9 for delta in score_deltas)}"
            ),
            "throughput geo %": format_geometric_change(tp_ratios),
            "TDR geo %": format_geometric_change(tdr_ratios),
            "TPOT geo %": format_geometric_change(tpot_ratios),
            "elapsed geo %": format_geometric_change(elapsed_ratios),
            "scheduler CPU ms": f"{scheduler_cpu_ms:.3f}",
        }
    )

display_table(summary_rows)

version,mean score,mean score delta,wins / ties / losses,throughput geo %,TDR geo %,TPOT geo %,elapsed geo %,scheduler CPU ms
v0-baseline,409.271,+0.000,0 / 29 / 0,+0.0%,+0.0%,+0.0%,+0.0%,741.543
v1-multi-active,485.012,+75.740,17 / 3 / 9,+23.5%,-76.2%,+116.2%,-19.0%,776.237
v2-load-aware,488.723,+79.451,18 / 4 / 7,+24.8%,-77.1%,+118.3%,-19.9%,813.458
v3-immediate-groups,612.911,+203.639,20 / 3 / 6,+73.8%,-77.8%,+38.6%,-42.5%,231.228
v4-table-groups,636.944,+227.673,20 / 3 / 6,+84.8%,-77.7%,+27.5%,-45.9%,240.513
v5-slo-aware,642.855,+233.583,20 / 2 / 7,+85.3%,-77.6%,+24.9%,-46.0%,239.593
v6-prefill-chunks,644.960,+235.689,22 / 2 / 5,+86.8%,-77.9%,+24.4%,-46.5%,239.548
v7-link-aware,660.868,+251.597,22 / 2 / 5,+85.9%,-78.9%,+33.6%,-46.2%,235.598
v8-exact-timelines,664.668,+255.396,22 / 2 / 5,+87.9%,-78.9%,+32.0%,-46.8%,235.499
v9-fanout-cohorts,665.625,+256.353,22 / 2 / 5,+88.3%,-78.9%,+32.0%,-46.9%,244.661


### Scenario score matrix

Green marks the best score observed for that scenario in this run. Ties are all highlighted.

In [11]:
score_matrix_rows = []
for scenario_name in selected_scenario_names:
    row: dict[str, Any] = {"scenario": scenario_name}
    for version in selected_versions:
        name = version["name"]
        row[name] = f"{result_map(name)[scenario_name]['score']:.3f}"
    score_matrix_rows.append(row)


def best_score_style(row: dict[str, Any], key: str) -> str:
    if key == "scenario":
        return "font-weight: 600"
    numeric_scores = [float(row[version["name"]]) for version in selected_versions]
    return "background: #d9f2df; font-weight: 600" if math.isclose(float(row[key]), max(numeric_scores), abs_tol=5e-4) else ""


display_table(
    score_matrix_rows,
    [("scenario", "Scenario")] + [(version["name"], version["name"]) for version in selected_versions],
    cell_style=best_score_style,
)

Scenario,v0-baseline,v1-multi-active,v2-load-aware,v3-immediate-groups,v4-table-groups,v5-slo-aware,v6-prefill-chunks,v7-link-aware,v8-exact-timelines,v9-fanout-cohorts,v10-batch-placement,v11-score-slack,v12-deadline-chunks,v13-attained-service,v14-backpressure,v15-one-token-lookahead,v16-counterfactual-groups,v17-learned-group-ranker,v18-nonlinear-group-ranker,v19-terminal-dpost,v20-terminal-dproc,v84-exact-dpost-partition,v85-censored-completion-index,v86-objective-margin-rollout,v87-guarded-margin-rollout,v88-robust-portfolio-policy,working-tree
official_worked_example,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000,500.000
single_sanity,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111,754.111
two_cloud_parallel,776.939,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
output_length_skew,724.137,695.922,696.830,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206,708.206
batch_friendly_burst,200.433,178.692,178.692,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,676.030,599.416,676.030,676.030,676.030
latency_sensitive_stream,591.985,694.482,692.204,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.673,996.029,987.995,996.673,996.673
link_bottleneck,243.119,241.116,241.116,241.116,241.116,241.116,241.109,241.109,241.109,241.109,241.109,241.109,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004,241.004
prefill_preemption,704.428,639.597,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802,730.802
degenerate_one_layer,766.371,997.324,997.324,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000,1000.000
interpolation_missing_values,761.047,878.791,878.791,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952,879.952


### Score deltas from the frozen baseline

In [12]:
delta_rows = []
for scenario_name in selected_scenario_names:
    row = {"scenario": scenario_name}
    for version in selected_versions:
        name = version["name"]
        delta = result_map(name)[scenario_name]["score"] - reference_map[scenario_name]["score"]
        row[name] = f"{delta:+.3f}"
    delta_rows.append(row)


def delta_style(row: dict[str, Any], key: str) -> str:
    if key == "scenario":
        return "font-weight: 600"
    value = float(row[key])
    if value > 1e-9:
        intensity = min(0.35, 0.08 + abs(value) / 500.0)
        return f"background: rgba(40, 167, 69, {intensity:.3f})"
    if value < -1e-9:
        intensity = min(0.35, 0.08 + abs(value) / 500.0)
        return f"background: rgba(220, 53, 69, {intensity:.3f})"
    return "background: #f3f4f6"


display_table(
    delta_rows,
    [("scenario", "Scenario")] + [(version["name"], version["name"]) for version in selected_versions],
    cell_style=delta_style,
)

Scenario,v0-baseline,v1-multi-active,v2-load-aware,v3-immediate-groups,v4-table-groups,v5-slo-aware,v6-prefill-chunks,v7-link-aware,v8-exact-timelines,v9-fanout-cohorts,v10-batch-placement,v11-score-slack,v12-deadline-chunks,v13-attained-service,v14-backpressure,v15-one-token-lookahead,v16-counterfactual-groups,v17-learned-group-ranker,v18-nonlinear-group-ranker,v19-terminal-dpost,v20-terminal-dproc,v84-exact-dpost-partition,v85-censored-completion-index,v86-objective-margin-rollout,v87-guarded-margin-rollout,v88-robust-portfolio-policy,working-tree
official_worked_example,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000
single_sanity,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000,+0.000
two_cloud_parallel,+0.000,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061,+223.061
output_length_skew,+0.000,-28.215,-27.307,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931,-15.931
batch_friendly_burst,+0.000,-21.741,-21.741,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+475.597,+398.984,+475.597,+475.597,+475.597
latency_sensitive_stream,+0.000,+102.497,+100.219,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.688,+404.044,+396.010,+404.688,+404.688
link_bottleneck,+0.000,-2.002,-2.002,-2.002,-2.002,-2.002,-2.010,-2.010,-2.010,-2.010,-2.010,-2.010,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115,-2.115
prefill_preemption,+0.000,-64.831,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374,+26.374
degenerate_one_layer,+0.000,+230.953,+230.953,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629,+233.629
interpolation_missing_values,+0.000,+117.744,+117.744,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905,+118.905


## 6. Isolate the effect of each layer

Versions 1–18 contain layers `1...N`. Comparing them with `v(N-1)` isolates the newly enabled
feature gate while keeping source, compiler flags, and scenarios fixed. Version 19 deliberately
branches from v15, while version 20 extends that terminal branch from v19.
These are deterministic local workload results—not statistical estimates and not a claim
about the official hidden-test distribution.

In [13]:
frozen_layers = sorted(
    (
        version
        for version in selected_versions
        if version.get("frozen")
        and (match := re.match(r"^v(\d+)-", version["name"])) is not None
        and int(match.group(1)) <= 20
    ),
    key=lambda version: version.get("layer", 0),
)
frozen_by_name = {version["name"]: version for version in frozen_layers}
lineage_predecessors = {"v19-terminal-dpost": "v15-one-token-lookahead"}
adjacent_layer_pairs = []
for current in frozen_layers:
    current_layer = current.get("layer", 0)
    if current_layer == 0:
        continue
    previous_name = lineage_predecessors.get(current["name"])
    if previous_name is None:
        previous_name = next(
            (
                candidate["name"]
                for candidate in frozen_layers
                if candidate.get("layer", 0) == current_layer - 1
            ),
            None,
        )
    if previous_name in frozen_by_name:
        adjacent_layer_pairs.append((frozen_by_name[previous_name], current))

incremental_rows = []
for previous_version, current_version in adjacent_layer_pairs:
    previous_name = previous_version["name"]
    current_name = current_version["name"]
    previous_results = result_map(previous_name)
    current_results = result_map(current_name)
    deltas = {
        scenario_name: (
            current_results[scenario_name]["score"]
            - previous_results[scenario_name]["score"]
        )
        for scenario_name in selected_scenario_names
    }
    best_scenario = max(deltas, key=deltas.get)
    worst_scenario = min(deltas, key=deltas.get)
    mean_previous = sum(
        previous_results[name]["score"] for name in selected_scenario_names
    ) / len(selected_scenario_names)
    mean_current = sum(
        current_results[name]["score"] for name in selected_scenario_names
    ) / len(selected_scenario_names)
    incremental_rows.append(
        {
            "transition": f"{previous_name} → {current_name}",
            "new policy": current_version["description"],
            "mean score": f"{mean_current:.3f}",
            "incremental mean": f"{mean_current - mean_previous:+.3f}",
            "wins / ties / losses": (
                f"{sum(delta > 1e-9 for delta in deltas.values())} / "
                f"{sum(abs(delta) <= 1e-9 for delta in deltas.values())} / "
                f"{sum(delta < -1e-9 for delta in deltas.values())}"
            ),
            "largest gain": f"{best_scenario} ({deltas[best_scenario]:+.3f})",
            "largest regression": f"{worst_scenario} ({deltas[worst_scenario]:+.3f})",
        }
    )

if incremental_rows:
    display_table(incremental_rows)
else:
    display(Markdown("_Select adjacent frozen layers to populate this table._"))

transition,new policy,mean score,incremental mean,wins / ties / losses,largest gain,largest regression
v0-baseline → v1-multi-active,Multiple unfinished singleton requests per cloud with round-robin assignment.,485.012,+75.740,17 / 3 / 9,attained_service_tail (+657.403),terminal_dpost_future_arrival (-624.596)
v1-multi-active → v2-load-aware,"Adds cloud assignment using busy time, known prefill work, ready decode work, and an active-request proxy.",488.723,+3.711,4 / 23 / 2,prefill_preemption (+91.204),latency_sensitive_stream (-2.278)
v2-load-aware → v3-immediate-groups,"Adds immediate grouping for ready D PRE, same-cloud D PROC, and D POST work.",612.911,+124.188,18 / 9 / 2,scheduler_cpu_stress (+855.329),attained_service_tail (-110.141)
v3-immediate-groups → v4-table-groups,Selects decode group size by interpolated task-table service rate instead of always taking every ready member.,636.944,+24.034,6 / 22 / 1,nonmonotonic_batch_table (+349.362),terminal_dproc_clearance (-12.383)
v4-table-groups → v5-slo-aware,Adds conservative SLO urgency and tightly gated controlled waiting when future events are known.,642.855,+5.910,2 / 24 / 3,cross_cloud_fanout (+406.480),one_token_lookahead (-125.108)
v5-slo-aware → v6-prefill-chunks,Adds adaptive gap-free P PROC layer chunks when longer prefills compete with other cloud work.,644.960,+2.105,3 / 25 / 1,single_cloud_prefill_interleave (+23.964),link_bottleneck (-0.007)
v6-prefill-chunks → v7-link-aware,Adds transfer-aware group cost and latency-weighted shortest-prefill ordering while preserving FIFO on throughput-weighted links.,660.868,+15.908,4 / 22 / 3,counterfactual_grouping (+355.939),nonlinear_ranker_holdout (-13.435)
v7-link-aware → v8-exact-timelines,Tracks exact virtual resource and FIFO-link finish times and bounds batching waits by the next known event.,664.668,+3.800,1 / 28 / 0,exact_wait_horizon (+110.188),official_worked_example (+0.000)
v8-exact-timelines → v9-fanout-cohorts,Adds cloud-fanout-aware D PRE group formation and waits only for compatible decode cohort events.,665.625,+0.957,2 / 27 / 0,exact_wait_horizon (+19.651),official_worked_example (+0.000)
v9-fanout-cohorts → v10-batch-placement,Adds conservative pack-versus-spread cloud placement using future same-cloud decode batching affinity.,667.429,+1.804,1 / 28 / 0,batch_aware_placement (+52.320),official_worked_example (+0.000)


### Did each mechanism move its isolation scenario?

These scenarios were written to make one scheduling pressure conspicuous. A positive delta
supports the narrow statement that the newly enabled mechanism helped on that constructed
input. A zero or negative result is equally useful: it tells us the heuristic did not buy
anything under that pressure or traded away a more valuable metric.

In [14]:
layer_targets = {
    "v1-multi-active": ["two_cloud_parallel"],
    "v2-load-aware": ["output_length_skew"],
    "v3-immediate-groups": ["batch_friendly_burst"],
    "v4-table-groups": ["nonmonotonic_batch_table"],
    "v5-slo-aware": ["slo_priority_collision"],
    "v6-prefill-chunks": ["single_cloud_prefill_interleave"],
    "v7-link-aware": ["latency_weighted_slow_link"],
    "v8-exact-timelines": ["exact_wait_horizon"],
    "v9-fanout-cohorts": ["cross_cloud_fanout"],
    "v10-batch-placement": ["batch_aware_placement"],
    "v11-score-slack": ["predicted_deadline_slack"],
    "v12-deadline-chunks": ["chunk_deadline_collision"],
    "v13-attained-service": ["attained_service_tail"],
    "v14-backpressure": ["downstream_backpressure"],
    "v15-one-token-lookahead": ["one_token_lookahead"],
    "v16-counterfactual-groups": ["counterfactual_grouping"],
    "v17-learned-group-ranker": ["learned_grouping_recovery"],
    "v18-nonlinear-group-ranker": ["nonlinear_ranker_holdout"],
    "v19-terminal-dpost": ["terminal_dpost_remainder"],
    "v20-terminal-dproc": ["terminal_dproc_clearance"],
}

target_rows = []
for previous_version, current_version in adjacent_layer_pairs:
    previous_name = previous_version["name"]
    current_name = current_version["name"]
    for scenario_name in layer_targets.get(current_name, []):
        if scenario_name not in selected_scenario_names:
            continue
        old = result_map(previous_name)[scenario_name]
        new = result_map(current_name)[scenario_name]
        target_rows.append(
            {
                "layer": current_name,
                "isolation scenario": scenario_name,
                "score before": f"{old['score']:.3f}",
                "score after": f"{new['score']:.3f}",
                "score delta": f"{new['score'] - old['score']:+.3f}",
                "throughput delta %": f"{100 * (new['throughput'] / old['throughput'] - 1):+.1f}%",
                "TDR delta %": f"{100 * (new['tdr'] / old['tdr'] - 1):+.1f}%" if old["tdr"] else "n/a",
                "TPOT delta %": f"{100 * (new['tpot'] / old['tpot'] - 1):+.1f}%" if old["tpot"] else "n/a",
            }
        )

display_table(target_rows)

layer,isolation scenario,score before,score after,score delta,throughput delta %,TDR delta %,TPOT delta %
v1-multi-active,two_cloud_parallel,776.939,1000.000,+223.061,+78.2%,-65.0%,+43.2%
v2-load-aware,output_length_skew,695.922,696.830,+0.907,+0.3%,+0.0%,-1.2%
v3-immediate-groups,batch_friendly_burst,178.692,676.030,+497.338,+332.7%,+0.0%,-86.6%
v4-table-groups,nonmonotonic_batch_table,650.638,1000.000,+349.362,+130.8%,+0.0%,-69.1%
v5-slo-aware,slo_priority_collision,920.706,923.650,+2.944,+2.0%,+3.1%,-1.4%
v6-prefill-chunks,single_cloud_prefill_interleave,649.866,673.830,+23.964,+10.2%,-33.2%,-7.7%
v7-link-aware,latency_weighted_slow_link,777.555,906.241,+128.686,+0.3%,-64.2%,+312.4%
v8-exact-timelines,exact_wait_horizon,281.588,391.776,+110.188,+34.5%,+1.1%,-28.8%
v9-fanout-cohorts,cross_cloud_fanout,991.899,1000.000,+8.101,+2.6%,+0.0%,+0.5%
v10-batch-placement,batch_aware_placement,105.698,158.018,+52.320,+41.5%,+17.6%,-38.1%


## 7. Focus on one scenario

Change `FOCUS_SCENARIO` in the parameter cell to inspect a different workload.

In [15]:
active_focus_scenario = (
    FOCUS_SCENARIO if FOCUS_SCENARIO in selected_scenario_names else selected_scenario_names[0]
)
if active_focus_scenario != FOCUS_SCENARIO:
    print(
        f"Configured focus scenario {FOCUS_SCENARIO!r} was filtered out; "
        f"showing {active_focus_scenario!r} instead."
    )
focus_rows = []
for version in selected_versions:
    name = version["name"]
    result = result_map(name)[active_focus_scenario]
    focus_rows.append(
        {
            "version": name,
            "score": f"{result['score']:.3f}",
            "score delta": f"{result['score'] - reference_map[active_focus_scenario]['score']:+.3f}",
            "throughput": f"{result['throughput']:.6f}",
            "TDR ms": f"{result['tdr']:.3f}",
            "TPOT ms": f"{result['tpot']:.3f}",
            "elapsed ms": f"{result['elapsed']:.3f}",
            "frames": result["frames"],
            "scheduler CPU ms": f"{1000 * result.get('scheduler_cpu_seconds', 0.0):.3f}",
        }
    )

display(
    Markdown(
        f"### `{active_focus_scenario}`\n\n"
        f"{scenario_data[active_focus_scenario]['description']}"
    )
)
display_table(focus_rows)

### `batch_friendly_burst`

A high-overhead request burst where decode grouping should materially improve throughput.

version,score,score delta,throughput,TDR ms,TPOT ms,elapsed ms,frames,scheduler CPU ms
v0-baseline,200.433,+0.000,0.052597,969.725,68.663,2433.600,721,10.929
v1-multi-active,178.692,-21.741,0.052597,207.931,270.324,2433.600,721,11.354
v2-load-aware,178.692,-21.741,0.052597,207.931,270.324,2433.600,721,13.738
v3-immediate-groups,676.030,+475.597,0.227611,208.020,36.179,562.363,217,6.575
v4-table-groups,676.030,+475.597,0.227611,208.020,36.179,562.363,217,5.615
v5-slo-aware,676.030,+475.597,0.227611,208.020,36.179,562.363,217,6.493
v6-prefill-chunks,676.030,+475.597,0.227611,208.020,36.179,562.363,217,6.555
v7-link-aware,676.030,+475.597,0.227611,208.020,36.179,562.363,217,5.898
v8-exact-timelines,676.030,+475.597,0.227611,208.020,36.179,562.363,217,5.601
v9-fanout-cohorts,676.030,+475.597,0.227611,208.020,36.179,562.363,217,6.264


## 8. Regression alerts

These alerts surface tradeoffs for review. They are not blanket rejection rules because a
throughput-oriented policy can rationally trade latency in one scenario for a larger score
gain elsewhere, depending on the supplied weights.

In [16]:
alert_rows = []
for version in selected_versions:
    name = version["name"]
    if name == REFERENCE_VERSION:
        continue
    current = result_map(name)
    for scenario_name in selected_scenario_names:
        old = reference_map[scenario_name]
        new = current[scenario_name]
        score_delta = new["score"] - old["score"]
        throughput_percent = 100.0 * (new["throughput"] / old["throughput"] - 1.0)
        tdr_percent = 100.0 * (new["tdr"] / old["tdr"] - 1.0) if old["tdr"] else 0.0
        tpot_percent = 100.0 * (new["tpot"] / old["tpot"] - 1.0) if old["tpot"] else 0.0
        reasons = []
        if score_delta < SCORE_REGRESSION_ALERT:
            reasons.append(f"score {score_delta:+.3f}")
        if throughput_percent < THROUGHPUT_REGRESSION_ALERT_PERCENT:
            reasons.append(f"throughput {throughput_percent:+.1f}%")
        if tdr_percent > LATENCY_REGRESSION_ALERT_PERCENT:
            reasons.append(f"TDR {tdr_percent:+.1f}%")
        if tpot_percent > LATENCY_REGRESSION_ALERT_PERCENT:
            reasons.append(f"TPOT {tpot_percent:+.1f}%")
        if reasons:
            alert_rows.append(
                {"version": name, "scenario": scenario_name, "review": ", ".join(reasons)}
            )

if alert_rows:
    display_table(alert_rows)
else:
    display(Markdown("_No configured regression thresholds were crossed._"))

version,scenario,review
v1-multi-active,two_cloud_parallel,TPOT +43.2%
v1-multi-active,output_length_skew,"score -28.215, throughput -7.3%, TPOT +21.4%"
v1-multi-active,batch_friendly_burst,"score -21.741, TPOT +293.7%"
v1-multi-active,latency_sensitive_stream,TPOT +199.3%
v1-multi-active,link_bottleneck,"score -2.002, TPOT +63.4%"
v1-multi-active,prefill_preemption,"score -64.831, throughput -20.4%"
v1-multi-active,interpolation_missing_values,TPOT +13.8%
v1-multi-active,single_cloud_prefill_interleave,TPOT +191.6%
v1-multi-active,slo_priority_collision,"score -60.682, TPOT +112.1%"
v1-multi-active,latency_weighted_slow_link,TPOT +173.8%


## 9. Save or export this run

Disposable per-version JSON is always written to `build/benchmark-runs/`. Set
`SAVE_DURABLE_RUN = True` only when the experiment is worth preserving in
`benchmarks/runs/`; existing run labels are never overwritten.

In [17]:
combined_run = {
    "schema_version": 1,
    "reference_version": REFERENCE_VERSION,
    "scenario_names": selected_scenario_names,
    "versions": version_rows,
    "results": results_by_version,
    "summary": summary_rows,
    "incremental_layers": incremental_rows,
    "isolation_scenarios": target_rows,
    "validation": {
        "all_legal": all(row["status"] == "PASS" for row in run_rows),
        "score_recomputed": True,
        "baseline_snapshot_max_delta": maximum_snapshot_delta,
    },
}

latest_path = RUN_DIR / "latest-combined.json"
latest_path.write_text(json.dumps(combined_run, indent=2) + "\n")
print(f"Wrote disposable combined result: {latest_path.relative_to(REPO_ROOT)}")

if SAVE_DURABLE_RUN:
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", DURABLE_RUN_LABEL)
    durable_dir = REPO_ROOT / "benchmarks/runs"
    durable_dir.mkdir(parents=True, exist_ok=True)
    durable_path = durable_dir / f"{DURABLE_RUN_LABEL}.json"
    if durable_path.exists():
        raise FileExistsError(f"Refusing to overwrite {durable_path}")
    durable_path.write_text(json.dumps(combined_run, indent=2) + "\n")
    print(f"Saved durable run: {durable_path.relative_to(REPO_ROOT)}")
else:
    print("Durable save disabled; set SAVE_DURABLE_RUN=True to preserve a named experiment.")

Wrote disposable combined result: build/benchmark-runs/latest-combined.json
Durable save disabled; set SAVE_DURABLE_RUN=True to preserve a named experiment.


## Checks

In [18]:
assert all(row["status"] == "PASS" for row in build_rows)
assert all(row["status"] == "PASS" for row in run_rows)
assert all(row["token counts"] == "verified" for row in validation_rows)
assert latest_path.is_file()
if VERSIONS_TO_RUN == "all" and SCENARIOS_TO_RUN == "all":
    assert len(incremental_rows) == 20
    assert len(target_rows) == 20
print("Benchmark workbench checks passed.")

Benchmark workbench checks passed.


## Next steps

Use the isolation table to choose the next experiment. Tune only one layer at a time, rerun
the complete workbench, and inspect both its target-scenario gain and its worst regression.
If a future policy becomes a meaningful checkpoint, preserve it with
`tools/register_scheduler.py` before continuing so the comparison remains reproducible.